In [15]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset


In [18]:
def create_labels_csv(data_dir):
    labels = []
    count = 1  # To keep track of images
    
    for i in range(16):  # Assuming 16 people in the dataset
        if i % 2 == 0:
            for pose in ['front', 'left', 'right']:  # Assign labels for each pose
                    labels.append([f'{count}.jpg', 'criminal'])  
                    count += 1
        else:
            for pose in ['front', 'left', 'right']:  # Assign labels for each pose
                labels.append([f'{count}.jpg', 'not criminal']) 
                count += 1
    
    df = pd.DataFrame(labels, columns=['image_name', 'label'])
    df.to_csv(os.path.join(data_dir, 'labels.csv'), index=False)

data_dir = 'D:\projects\Facial Recongniton\\'  # Update to your actual dataset directory
create_labels_csv(data_dir)


## Data Preprocessing

In [19]:
class FaceDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.labels_df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.labels_df.iloc[idx, 0])
        image = cv2.imread(img_name)
        
        # Check if the image was loaded successfully
        if image is None:
            raise FileNotFoundError(f"Image not found: {img_name}")
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = 1 if self.labels_df.iloc[idx, 1] == 'criminal' else 0
    
        if self.transform:
            image = self.transform(image)
    
        return image, label

# Define transformations (resize, normalize)
data_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Standard normalization for pretrained models
])

dataset = FaceDataset(csv_file=os.path.join(data_dir, 'labels.csv'), root_dir=data_dir, transform=data_transforms)
train_set, val_set = train_test_split(dataset, test_size=0.2, random_state=42)

train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False)


## Pretrained Model Setup

In [20]:
model = models.mobilenet_v2(pretrained=True)

# Replace the last layer for binary classification (criminal vs. non-criminal)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 1)  # Binary classification

# Move model to GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)


C:\Users\shafq\.conda\envs\fyp-tensorflow\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\shafq\.conda\envs\fyp-tensorflow\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## Training Pipeline

In [21]:
# Define loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).float().unsqueeze(1)  # Convert to float and adjust shape for BCE loss

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')


Epoch [1/10], Loss: 0.8235
Epoch [2/10], Loss: 0.2453
Epoch [3/10], Loss: 0.0734
Epoch [4/10], Loss: 0.4691
Epoch [5/10], Loss: 0.7018
Epoch [6/10], Loss: 0.1802
Epoch [7/10], Loss: 0.3344
Epoch [8/10], Loss: 0.0862
Epoch [9/10], Loss: 0.0594
Epoch [10/10], Loss: 0.0252


## Validation

In [22]:
model.eval()
corrects = 0
total = 0

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        
        outputs = model(inputs)
        preds = torch.sigmoid(outputs) > 0.5
        
        corrects += torch.sum(preds == labels).item()
        total += labels.size(0)

print(f'Validation Accuracy: {corrects / total:.4f}')


Validation Accuracy: 0.7000


In [25]:
torch.save(model.state_dict(), "D:\projects\Facial Recongniton\70%_mobile_net_model.pth")

## Prediction

In [28]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import os
import torchvision.models as models
import torch.nn as nn

# Function to load a trained model
def load_model(model_path):
    # Define the model architecture
    model = models.mobilenet_v2(pretrained=True)
    
    # Replace the last layer for binary classification (criminal vs. non-criminal)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, 1)  # Binary classification

    # Load the model weights
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()  # Set the model to evaluation mode
    return model

# Function to preprocess the input image
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize the image to the input size of the model
        transforms.ToTensor(),  # Convert the image to tensor
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize using ImageNet statistics
    ])
    
    image = Image.open(image_path).convert('RGB')  # Open the image and convert to RGB
    image = transform(image)  # Apply the transformation
    image = image.unsqueeze(0)  # Add a batch dimension
    return image

# Function to make a prediction on a single image
def predict_image(model, image_tensor):
    with torch.no_grad():  # Disable gradient computation for inference
        outputs = model(image_tensor)  # Forward pass
        # Apply sigmoid to get probability and then threshold to get binary prediction
        probability = torch.sigmoid(outputs)
        predicted = (probability > 0.5).int()  # Convert probabilities to binary (0 or 1)
    return predicted

# Load the trained model (replace with your trained model path)
model_path = 'D:\\projects\\Facial Recongniton\\70%_mobile_net_model.pth'
model = load_model(model_path)

# Image for prediction (replace with your image path)
image_path = 'D:\\projects\\Facial Recongniton\\test\\test_image.jpg'  # Example test image path

# Preprocess the image
image_tensor = preprocess_image(image_path)

# Get prediction
predicted_label = predict_image(model, image_tensor)

# Convert the prediction to human-readable format
class_names = ['not criminal', 'criminal']  # Update based on your dataset
predicted_class = class_names[predicted_label.item()]

# Display the result
print(f'Prediction for the image: {predicted_class}')


Prediction for the image: not criminal


C:\Users\shafq\AppData\Local\Temp\ipykernel_14148\2675110371.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=d